In [1]:
# DAY 4: MODEL TRAINING (IMPROVED)
# UCI Heart Disease Prediction
# PURPOSE:
#   - Load leakage-safe processed train/test data from Day 3
#   - Train a broad set of beginner-friendly and advanced ML models
#   - Save every successfully trained model
#   - Save a model manifest for Day 5 evaluation

import pandas as pd
import numpy as np
import json
import os
import sys
from datetime import datetime

sys.path.append('scripts')
from utils import build_model_zoo, save_model, save_json, RANDOM_STATE


In [2]:
# SECTION 1: LOAD PREPROCESSED TRAIN/TEST DATA

print("LOADING PREPROCESSED DATA")

# Day 3 already split the data and fitted preprocessing on the training set only.
X_train = pd.read_csv('data/processed/X_train_processed.csv')
X_test = pd.read_csv('data/processed/X_test_processed.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

print(f"Training feature shape: {X_train.shape}")
print(f"Testing feature shape:  {X_test.shape}")
print(f"Training target shape:  {y_train.shape}")
print(f"Testing target shape:   {y_test.shape}")
print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())


LOADING PREPROCESSED DATA
Training feature shape: (736, 29)
Testing feature shape:  (184, 29)
Training target shape:  (736,)
Testing target shape:   (184,)

Training class distribution:
num
0    329
1    212
2     87
3     86
4     22
Name: count, dtype: int64


In [3]:
# SECTION 2: VERIFY TRAIN-TEST SPLIT

print("VERIFYING TRAIN-TEST SPLIT")

print("✓ Train/test split was created in Day 3 before preprocessing")
print("✓ Day 4 will train only on the processed training set")
print("✓ Test set is kept untouched until Day 5 evaluation")

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")


VERIFYING TRAIN-TEST SPLIT
✓ Train/test split was created in Day 3 before preprocessing
✓ Day 4 will train only on the processed training set
✓ Test set is kept untouched until Day 5 evaluation
Training set: (736, 29)
Testing set:  (184, 29)


In [4]:
# SECTION 3: DEFINE MODELS

print("DEFINING MODELS")

# Preprocessing has already been fitted safely in Day 3.
# This model zoo intentionally includes many algorithms for comparison.
MODELS = build_model_zoo(random_state=RANDOM_STATE)

print(f"Models defined: {len(MODELS)}")
for model_name in MODELS:
    print(f"  • {model_name}")


DEFINING MODELS
Models defined: 22
  • dummy_baseline
  • logistic_regression
  • ridge_classifier
  • sgd_classifier
  • perceptron
  • knn
  • gaussian_nb
  • linear_svm
  • svm_rbf
  • decision_tree
  • random_forest
  • extra_trees
  • bagging
  • adaboost
  • gradient_boosting
  • hist_gradient_boosting
  • linear_discriminant_analysis
  • nearest_centroid
  • xgboost
  • lightgbm
  • catboost
  • random_forest_random_search


In [5]:
# SECTION 4: TRAIN ALL MODELS

print("TRAINING MODELS")

trained_models = {}
training_log = []

for model_name, model in MODELS.items():
    try:
        print(f"\n▶ Training {model_name}...", end=" ")
        model.fit(X_train, y_train)
        trained_models[model_name] = model
        training_log.append({
            'model': model_name,
            'status': 'success',
            'timestamp': datetime.now().isoformat()
        })
        print("✓ Done")
    except Exception as e:
        training_log.append({
            'model': model_name,
            'status': 'failed',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        })
        print(f"✗ Failed: {str(e)[:80]}")

print(f"\n✓ Successfully trained {len(trained_models)}/{len(MODELS)} models")


TRAINING MODELS

▶ Training dummy_baseline... ✓ Done

▶ Training logistic_regression... ✓ Done

▶ Training ridge_classifier... ✓ Done

▶ Training sgd_classifier... ✓ Done

▶ Training perceptron... ✓ Done

▶ Training knn... ✓ Done

▶ Training gaussian_nb... ✓ Done

▶ Training linear_svm... ✓ Done

▶ Training svm_rbf... ✓ Done

▶ Training decision_tree... ✓ Done

▶ Training random_forest... ✓ Done

▶ Training extra_trees... ✓ Done

▶ Training bagging... ✓ Done

▶ Training adaboost... ✓ Done

▶ Training gradient_boosting... ✓ Done

▶ Training hist_gradient_boosting... ✓ Done

▶ Training linear_discriminant_analysis... ✓ Done

▶ Training nearest_centroid... ✓ Done

▶ Training xgboost... ✓ Done

▶ Training lightgbm... ✓ Done

▶ Training catboost... ✓ Done

▶ Training random_forest_random_search... ✓ Done

✓ Successfully trained 22/22 models


In [6]:
# SECTION 5: SAVE TRAINED MODELS

print("SAVING TRAINED MODELS (FOR EVALUATION)")

os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/reports', exist_ok=True)

model_manifest = {}

for model_name, model in trained_models.items():
    model_path = f'outputs/models/{model_name}_trained.pkl'
    save_model(model, model_path)
    model_manifest[model_name] = model_path
    print(f"✓ Saved: {model_name}")

save_json(model_manifest, 'outputs/reports/day4_model_manifest.json')
print("✓ Saved: outputs/reports/day4_model_manifest.json")


SAVING TRAINED MODELS (FOR EVALUATION)
✓ Saved: dummy_baseline
✓ Saved: logistic_regression
✓ Saved: ridge_classifier
✓ Saved: sgd_classifier
✓ Saved: perceptron
✓ Saved: knn
✓ Saved: gaussian_nb
✓ Saved: linear_svm
✓ Saved: svm_rbf
✓ Saved: decision_tree
✓ Saved: random_forest
✓ Saved: extra_trees
✓ Saved: bagging
✓ Saved: adaboost
✓ Saved: gradient_boosting
✓ Saved: hist_gradient_boosting
✓ Saved: linear_discriminant_analysis
✓ Saved: nearest_centroid
✓ Saved: xgboost
✓ Saved: lightgbm
✓ Saved: catboost
✓ Saved: random_forest_random_search
  Saved: outputs/reports/day4_model_manifest.json
✓ Saved: outputs/reports/day4_model_manifest.json


In [7]:
# SECTION 6: TRAINING SUMMARY & EXPORT

print("TRAINING SUMMARY")

summary = {
    'training_date': datetime.now().isoformat(),
    'total_models_defined': len(MODELS),
    'total_models_trained': len(trained_models),
    'models_trained': list(trained_models.keys()),
    'training_log': training_log,
    'data_split': {
        'train_samples': int(X_train.shape[0]),
        'test_samples': int(X_test.shape[0]),
        'features': int(X_train.shape[1]),
        'split_before_preprocessing': True,
        'random_state': RANDOM_STATE
    }
}

save_json(summary, 'outputs/reports/day4_training_summary.json')
print("✓ Saved: day4_training_summary.json")


TRAINING SUMMARY
  Saved: outputs/reports/day4_training_summary.json
✓ Saved: day4_training_summary.json


In [8]:
# Save test split for Day 5

os.makedirs('outputs', exist_ok=True)

# Day 5 currently loads the test split from outputs/, so keep these copies there.
X_test.to_csv('outputs/X_test.csv', index=False)
y_test.to_csv('outputs/y_test.csv', index=False)

print("✓ Saved test split for Day 5")


✓ Saved test split for Day 5
